# Clase 221 — Cold-start: popularity Bayesiana, onboarding, bandits

Estrategias para users/items sin historia. Demos sintéticos.

In [ ]:
import numpy as np, pandas as pd

# Items con n_ratings y mean_rating distintos
items = pd.DataFrame({
    'item': range(1, 11),
    'n':    [2, 5, 10, 50, 100, 500, 1000, 1500, 3, 1],
    'mean': [5.0, 4.8, 4.5, 4.4, 4.3, 4.2, 4.1, 4.0, 5.0, 5.0],
})
items['naive_score'] = items['mean']

# Popularity vanilla: ordenar por mean → ganan items con 1-2 ratings 5/5 (overconfident)
print('--- Popularity vanilla (por mean) ---')
print(items.sort_values('naive_score', ascending=False).head(5).to_string(index=False))

## 1. Bayesian shrinkage

In [ ]:
C = (items['mean'] * items['n']).sum() / items['n'].sum()
m = 50   # prior weight: equivale a 50 ratings al promedio global
items['bayes_score'] = (items['n'] * items['mean'] + m * C) / (items['n'] + m)

print(f'global mean C = {C:.3f}\n')
print('--- Bayesian shrinkage ---')
print(items.sort_values('bayes_score', ascending=False).head(5).to_string(index=False))
print('\n→ Items con n bajo (2, 1, 3) caen porque el prior los ancla a C.')
print('  Items con n alto y rating bueno (1000-1500) dominan.')

## 2. Onboarding: usuario nuevo elige géneros

In [ ]:
rng = np.random.default_rng(42)
n_items = 200
GENRES = ['action', 'comedy', 'drama', 'sci-fi', 'romance']
item_genres = pd.DataFrame({
    'item': range(n_items),
    'genre': rng.choice(GENRES, size=n_items),
    'popularity': rng.integers(10, 5000, n_items),
    'mean_rating': np.clip(rng.normal(3.8, 0.4, n_items), 1, 5),
})

def onboarding_recommendations(selected_genres, df, n=10):
    """User nuevo elige géneros → recomendar mejores items en esos géneros con Bayesian shrinkage."""
    df = df[df.genre.isin(selected_genres)].copy()
    C = df['mean_rating'].mean()
    m = 100
    df['score'] = (df['popularity'] * df['mean_rating'] + m * C) / (df['popularity'] + m)
    return df.nlargest(n, 'score')[['item', 'genre', 'popularity', 'mean_rating', 'score']]

print('User nuevo eligió: sci-fi, action')
print(onboarding_recommendations(['sci-fi', 'action'], item_genres).to_string(index=False))

## 3. Item cold-start con content-based

In [ ]:
# Agregamos 3 movies nuevas (0 ratings)
new_movies = pd.DataFrame([
    {'item': 999, 'genre': 'sci-fi', 'popularity': 0, 'mean_rating': np.nan},
    {'item': 998, 'genre': 'action', 'popularity': 0, 'mean_rating': np.nan},
    {'item': 997, 'genre': 'comedy', 'popularity': 0, 'mean_rating': np.nan},
])
all_items = pd.concat([item_genres, new_movies], ignore_index=True)

def cold_start_recs(selected_genres, df, n=10, novelty_boost=True):
    # Items sin historia: usar prior global como rating; boost por ser nuevo
    df = df.copy()
    C_global = df['mean_rating'].mean()
    is_new = df['mean_rating'].isna()
    df.loc[is_new, 'mean_rating'] = C_global
    df.loc[is_new, 'popularity'] = 100   # "weight" pequeño
    df['is_new'] = is_new

    df = df[df.genre.isin(selected_genres)]
    df['score'] = df['mean_rating'].copy()
    if novelty_boost:
        df.loc[df.is_new, 'score'] = df.loc[df.is_new, 'score'] + 0.5   # boost a items nuevos
    return df.nlargest(n, 'score')[['item', 'genre', 'popularity', 'mean_rating', 'is_new', 'score']]

print('Cold-start de items nuevos en sci-fi/action:')
print(cold_start_recs(['sci-fi', 'action'], all_items).to_string(index=False))
print('\n→ items_999 y 998 (nuevos) están en el top por el novelty boost.')

## 4. Epsilon-greedy bandit

In [ ]:
class EpsilonGreedy:
    def __init__(self, n_arms, eps=0.1, rng=None):
        self.n_arms = n_arms
        self.eps = eps
        self.rewards = np.zeros(n_arms)
        self.pulls = np.zeros(n_arms, dtype=int)
        self.rng = rng or np.random.default_rng(0)

    def select(self):
        if self.rng.random() < self.eps:
            return self.rng.integers(self.n_arms)
        avg = np.where(self.pulls > 0, self.rewards / np.maximum(self.pulls, 1), 0)
        return int(np.argmax(avg))

    def update(self, arm, reward):
        self.pulls[arm] += 1
        self.rewards[arm] += reward

# Simular 1000 visitas con 5 "items" cuya conversion real es distinta
true_conv = np.array([0.10, 0.05, 0.20, 0.08, 0.15])
bandit = EpsilonGreedy(n_arms=5, eps=0.1, rng=np.random.default_rng(1))

for _ in range(1000):
    arm = bandit.select()
    reward = int(np.random.random() < true_conv[arm])
    bandit.update(arm, reward)

print(f'{"arm":>4} {"true_conv":>10} {"pulls":>7} {"observed":>10}')
for a in range(5):
    obs = bandit.rewards[a] / max(bandit.pulls[a], 1)
    print(f'{a:>4} {true_conv[a]:>10.3f} {bandit.pulls[a]:>7d} {obs:>10.3f}')
print(f'\n→ arm 2 (true_conv 0.20) recibió la mayoría de pulls.')
print(f'  arm 1 (true_conv 0.05) recibió pocos — eps explora pero no estanca ahí.')

## Ejercicio guiado

1. Sobre MovieLens, simulá 100 users nuevos (sin historia). Compará NDCG@10 con: (a) popularity vanilla, (b) Bayesian shrinkage, (c) onboarding + content-based.
2. Para items nuevos: trackeá tiempo hasta acumular 10 ratings comparando estrategia "sin boost" vs "con novelty boost".
3. Thompson sampling vs epsilon-greedy: implementá Thompson sampling con priors Beta para conversiones binarias.
4. Contextual bandit: usa demographics del user como contexto (edad, país). Compará vs vanilla bandit.
5. Bonus: cross-domain transfer: importa "gustos" de un dataset (música) y prediga preferencias en otro (películas).

## Conclusiones

- Cold-start es PRODUCTO, no solo modelo: onboarding, novelty boost, fallbacks son decisiones de UX.
- Bayesian shrinkage es la fix más barata y efectiva para popularity ranking.
- Content features (Clase 218) son la llave para item cold-start.
- Bandits resuelven la exploración cuando offline metrics no alcanzan.

## ✅ Soluciones de los ejercicios

El *cold-start* es "¿qué recomiendo cuando no tengo historial?" — usuario nuevo, item nuevo,
o sistema nuevo. Las herramientas: popularidad (con shrinkage bayesiano), onboarding por
géneros, content-based para items nuevos, y exploración ε-greedy. Todo con numpy, sin
internet.

### Setup: catálogo con géneros, ratings y conteos

Clave: la **cantidad** de ratings (`R_count`) es independiente de la **calidad** real. Así
hay items con pocos ratings pero promedio inflado por suerte — justo lo que el shrinkage
bayesiano corrige.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n_users, n_items, n_gen = 200, 60, 5
generos = ["Action", "Sci-Fi", "Comedy", "Drama", "Horror"]
item_gen = rng.integers(0, n_gen, n_items)

R_count = rng.integers(20, 400, n_items)       # popularidad (soporte)
R_count[:20] = rng.integers(1, 6, 20)          # 20 items "nicho": muy pocos ratings
true_quality = rng.uniform(2.5, 4.5, n_items)  # calidad real, INDEPENDIENTE del count
R_sum = np.array([rng.normal(true_quality[i], 1.0, R_count[i]).clip(1, 5).sum()
                  for i in range(n_items)])
mean_rating = R_sum.sum() / R_count.sum()
print("catálogo:", n_items, "items | rating global medio:", round(mean_rating, 2))
print("items nicho (n<6):", int((R_count < 6).sum()))
print("OK — catálogo listo")

### Ejercicio 1 — Popularity baseline

Sin datos del usuario, recomendamos lo más popular (más ratings). Es la mejor apuesta ciega y
el baseline obligado. El top-10 es el mismo para todo usuario cold-start.

In [ ]:
pop_rank = np.argsort(R_count)[::-1]
top10_pop = pop_rank[:10]
print("top-10 por popularidad (n_ratings):", top10_pop.tolist())

# evaluación: el gusto real de un user nuevo es un género; ¿el top popular acierta algo?
hits = []
for _ in range(50):
    g = rng.integers(0, n_gen)
    relevant = set(np.where(item_gen == g)[0])
    hits.append(len(set(top10_pop) & relevant) / min(10, len(relevant)))
print("recall@10 medio del popularity en users cold-start:", round(float(np.mean(hits)), 3))
assert len(top10_pop) == 10 and (R_count[top10_pop] >= 20).all()
print("OK ejercicio 1 — popularity baseline (mismo top-10 para todo cold-start)")

### Ejercicio 2 — Shrinkage bayesiano

`score = (sum + m·C) / (n + m)` con `m=10`, `C=media global`. Un item nicho con 3 ratings de
5.0 no debería ganarle a uno con 400 ratings de 4.4: el shrinkage empuja los de pocos datos
hacia la media.

In [ ]:
m, C = 10, mean_rating
avg_vanilla = R_sum / R_count
avg_bayes = (R_sum + m * C) / (R_count + m)

top_vanilla = np.argsort(avg_vanilla)[::-1][:10]
top_bayes = np.argsort(avg_bayes)[::-1][:10]
print("counts del top-10 vanilla:", R_count[top_vanilla].tolist())
print("counts del top-10 bayes  :", R_count[top_bayes].tolist())

# el promedio vanilla se deja engañar por items nicho; el bayes exige soporte
assert np.median(R_count[top_bayes]) > np.median(R_count[top_vanilla]),     "el shrinkage favorece items con más ratings"
solo = int(np.argmin(R_count))
print(f"item con menos ratings (n={R_count[solo]}): vanilla={avg_vanilla[solo]:.2f} -> "
      f"bayes={avg_bayes[solo]:.2f} (hacia C={C:.2f})")
assert abs(avg_bayes[solo] - C) < abs(avg_vanilla[solo] - C) + 1e-9
print("OK ejercicio 2 — shrinkage bayesiano estabiliza items con pocos ratings")

### Ejercicio 3 — Onboarding por 3 géneros

Un usuario nuevo elige `["Action", "Sci-Fi", "Comedy"]`. Recomendamos top-10 de esos géneros,
usando popularidad como desempate.

In [ ]:
elegidos = ["Action", "Sci-Fi", "Comedy"]
gid = [generos.index(g) for g in elegidos]
candidatos = np.where(np.isin(item_gen, gid))[0]
onboarding_top10 = candidatos[np.argsort(R_count[candidatos])[::-1]][:10]
print("top-10 onboarding:", onboarding_top10.tolist())
print("géneros:", [generos[item_gen[i]] for i in onboarding_top10])

assert len(onboarding_top10) == 10
assert all(item_gen[i] in gid for i in onboarding_top10), "todos de los géneros elegidos"
print("OK ejercicio 3 — onboarding por géneros con desempate por popularidad")

### Ejercicio 4 — Item cold-start

Agregamos 10 items nuevos con género pero **0 ratings**. La popularidad y el CF nunca los
mostrarán (n=0), pero el content-based sí los rankea por su género. Lo demostramos.

In [ ]:
new_items = np.arange(n_items, n_items + 10)
new_gen = rng.integers(0, n_gen, 10)
all_gen = np.concatenate([item_gen, new_gen])
all_count = np.concatenate([R_count, np.zeros(10, dtype=int)])   # 0 ratings

pop_all = all_count.astype(float)
top_pop_all = np.argsort(pop_all)[::-1][:10]
nuevos_en_pop = len(set(top_pop_all) & set(new_items.tolist()))

fav_gen = generos.index("Horror")
content_score = (all_gen == fav_gen).astype(float)
top_content = [i for i in np.argsort(content_score)[::-1] if content_score[i] > 0][:10]
nuevos_en_content = len(set(top_content) & set(new_items.tolist()))

print(f"items nuevos en top-10 popularidad: {nuevos_en_pop} (CF no los conoce)")
print(f"items nuevos en top-10 content    : {nuevos_en_content} (content sí los rankea)")
assert nuevos_en_pop == 0
assert nuevos_en_content >= 1, "content-based puede recomendar items sin historial"
print("OK ejercicio 4 — content resuelve el cold-start de items nuevos")

### Ejercicio 5 — Epsilon-greedy (explore/exploit)

En cada slot del top-10: con prob `ε=0.1` un item random (explore), con `0.9` el mejor del
modelo (exploit). Explorar levanta la **coverage** del catálogo a costa de algo de precisión.

In [ ]:
def epsilon_greedy_recs(base_ranking, n_total, eps=0.1, k=10, seed=0):
    r = np.random.default_rng(seed)
    recs, pool = [], list(base_ranking)
    for _ in range(k):
        if r.random() < eps:
            recs.append(int(r.integers(0, n_total)))         # explore
        else:
            for c in pool:
                if c not in recs:
                    recs.append(int(c)); break               # exploit
    return recs[:k]

def coverage(eps):
    shown = set()
    for u in range(100):
        shown.update(epsilon_greedy_recs(pop_rank, n_items, eps=eps, seed=u))
    return len(shown) / n_items

cov_greedy, cov_eps = coverage(0.0), coverage(0.1)
print(f"coverage ε=0.0 (solo exploit) = {cov_greedy:.1%}")
print(f"coverage ε=0.1 (con explore)  = {cov_eps:.1%}")
assert cov_eps > cov_greedy, "explorar aumenta la cobertura del catálogo"
print("OK ejercicio 5 — ε-greedy sube la coverage vía exploración")